In [14]:
using SpeedyWeather, CairoMakie, GLMakie

In [15]:
spectral_grid = SpectralGrid()

SpectralGrid{Spectrum{...}, OctahedralGaussianGrid{...}}
├ Number format: Float32
├ Spectral:      T31 LowerTriangularMatrix
├ Grid:          48-ring OctahedralGaussianGrid, 3168 grid points
├ Resolution:    3.61°, 401km (at 6371km radius)
├ Vertical:      8-layer atmosphere, 2-layer land
└ Architecture:  CPU using Array

In [16]:
model = PrimitiveWetModel(spectral_grid)
simulation = initialize!(model)

Simulation{PrimitiveWetModel}
├ prognostic_variables::PrognosticVariables{...}
├ diagnostic_variables::DiagnosticVariables{...}
└ model::PrimitiveWetModel{...}

## Output variables

In [17]:
model.output

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ v: meridional wind [m/s]
 ├ humid: specific humidity [kg/kg]
 ├ temp: temperature [degC]
 ├ u: zonal wind [m/s]
 ├ mslp: mean sea-level pressure [hPa]
 └ vor: relative vorticity [s^-1]

In [18]:
add!(model, SpeedyWeather.RadiationOutput()...)

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ sru: Surface shortwave radiation up [W/m^2]
 ├ temp: temperature [degC]
 ├ srd: Surface shortwave radiation down [W/m^2]
 ├ mslp: mean sea-level pressure [hPa]
 ├ vor: relative vorticity [s^-1]
 ├ osr: Outgoing shortwave radiation [W/m^2]
 ├ v: meridional wind [m/s]
 ├ u: zonal wind [m/s]
 ├ albedo: albedo [1]
 ├ lrd: Surface longwave radiation down [W/m^2]
 ├ humid: specific humidity [kg/kg]
 ├ olr: Outgoing longwave radiation [W/m^2]
 └ lru: Surfa

In [19]:
add!(model, SpeedyWeather.SurfaceFluxesOutput()...)

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ slf: Surface latent heat flux (positive up) [W/m^2]
 ├ sru: Surface shortwave radiation up [W/m^2]
 ├ temp: temperature [degC]
 ├ srd: Surface shortwave radiation down [W/m^2]
 ├ mslp: mean sea-level pressure [hPa]
 ├ vor: relative vorticity [s^-1]
 ├ osr: Outgoing shortwave radiation [W/m^2]
 ├ v: meridional wind [m/s]
 ├ u: zonal wind [m/s]
 ├ albedo: albedo [1]
 ├ lrd: Surface longwave radiation down [W/m^2]
 ├ shf: Surface sensible heat flux (po

In [20]:
simulation.diagnostic_variables.physics.sensible_heat_flux

3168-element, 48-ring OctahedralGaussianField{Float32, 1} as Array on CPU
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 ⋮
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0

In [21]:
simulation.diagnostic_variables.physics.surface_latent_heat_flux

3168-element, 48-ring OctahedralGaussianField{Float32, 1} as Array on CPU
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 ⋮
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0

In [22]:
# run!(simulation, period=Day(30))
run!(simulation, period=Day(20))

In [23]:
heatmap(simulation.diagnostic_variables.physics.sensible_heat_flux)

In [24]:
simulation.prognostic_variables.clock

Clock
├ time::DateTime = 2000-01-21T00:00:00
├ start::DateTime = 2000-01-01T00:00:00
├ period::Second = 1728000 seconds
├ timestep_counter::Int64 = 720
├ n_timesteps::Int64 = 720
└ Δt::Millisecond = 2400000 milliseconds

In [25]:
function calc_global_sum(field, model)
    a00 = real(transform(field)[1])
    mean_per_m2 = a00 / model.spectral_transform.norm_sphere
    total_W = mean_per_m2 * (4*pi*model.planet.radius^2)
    return total_W
end


calc_global_sum (generic function with 1 method)

In [26]:
function calc_global_mean(field, model)
    a00 = real(transform(field)[1])
    return a00 / model.spectral_transform.norm_sphere    
end
# mean_per_m2 = a00 / model.spectral_transform.norm_sphere

calc_global_mean (generic function with 1 method)

In [27]:
mean_SHF = calc_global_mean(simulation.diagnostic_variables.physics.sensible_heat_flux, model)

7.5233183f0

In [28]:
mean_LHF = calc_global_mean(simulation.diagnostic_variables.physics.surface_latent_heat_flux, model)

14.680991f0

In [29]:
simulation.diagnostic_variables.physics

PhysicsVariables
├ grid: OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}
├ ocean: DynamicsVariablesOcean{Float32, Array, OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Field{Float32, 1, Vector{Float32}, OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ land: DynamicsVariablesLand{Float32, Array, OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Field{Float32, 1, Vector{Float32}, OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ rain_large_scale: 3168-element, 48-ring Field{Float32}
├ rain_convection: 3168-element, 48-ring Field{Float32}
├ snow_large_scale: 3168-element, 48-ring Field{Float32}
├ snow_convection: 3168-element, 48-ring Field{Float32}
├ total_precipitation_rate: 3168-element, 48-ring Field{Float32}
├ cloud_top: 3168-element, 48-ring Field{Float32}

In [30]:
function calc_trenberth_variables(simulation, model; SumFlag::Bool=false)
    # this is a function to calculate the variables we need to plot the Trenberth diagram.
    # simulation -> the SpeedyWeather simulation output.
    # model -> the SpeedyWeather model structure.
    # SumFlag -> false for clculating using the area mean of the fluxes [W/m^2] and true for calculating for the global sum [W]. 

        # put all fields in a Dict
    fields = Dict(
        :LHF   => simulation.diagnostic_variables.physics.surface_latent_heat_flux,
        :SHF   => simulation.diagnostic_variables.physics.sensible_heat_flux,
        :SSRU  => simulation.diagnostic_variables.physics.surface_shortwave_up,
        :SLRU  => simulation.diagnostic_variables.physics.surface_longwave_up,
        :SSRD  => simulation.diagnostic_variables.physics.surface_shortwave_down,
        :SLRD  => simulation.diagnostic_variables.physics.surface_longwave_down,
        :OSR   => simulation.diagnostic_variables.physics.outgoing_shortwave_radiation,
        :OLR   => simulation.diagnostic_variables.physics.outgoing_longwave_radiation,
        :albedo => simulation.diagnostic_variables.physics.albedo
    )

    # initialize result container
    results = Dict{Symbol, Float64}()

    # pick the function once
    calcfun = SumFlag ? calc_global_sum : calc_global_mean

    for (name, field) in fields
        try
            results[name] = calcfun(field, model)
        catch err
            # more informative error handling
            @warn "Could not compute $name: $err"
            results[name] = NaN
        end
    end

    # optional derived Trenberth terms (example: ASR, surface net radiation)
    # note: sign conventions may vary in your model; adapt as needed
    # ASR (absorbed shortwave at TOA) = incoming_TOA - reflected_TOA
    # If you only have OSR as outgoing shortwave at TOA, and you know S_in_TOA:
    # results[:ASR] = S_in_global_total - results[:OSR]  # only if you have S_in_TOA
    
    # Simple surface net (down - up) :
    results[:SW_net_sfc] = results[:SSRD] - results[:SSRU]    # W/m2 or W
    results[:LW_net_sfc] = results[:SLRD] - results[:SLRU]
    results[:surface_net]  = results[:SW_net_sfc] + results[:LW_net_sfc] - results[:LHF] - results[:SHF]

    return results

    return results
end


calc_trenberth_variables (generic function with 1 method)

In [31]:
R_mean = calc_trenberth_variables(simulation, model; SumFlag=false)  # global means

Dict{Symbol, Float64} with 12 entries:
  :LW_net_sfc  => -242.621
  :OLR         => 322.865
  :SSRD        => 341.255
  :SLRD        => 0.0
  :SHF         => 7.52332
  :SSRU        => 26.767
  :SLRU        => 242.621
  :LHF         => 14.681
  :albedo      => 0.115555
  :SW_net_sfc  => 314.488
  :OSR         => 26.767
  :surface_net => 49.663

#### adding callbacks: 

In [32]:
# --------------------------
# helper: compute Trenberth diagnostics from diagn + model
# --------------------------
function calc_trenberth_from_diagn(diagn, model; SumFlag::Bool=false)
    fields = Dict(
        :LHF   => diagn.physics.surface_latent_heat_flux,
        :SHF   => diagn.physics.sensible_heat_flux,
        :SSRU  => diagn.physics.surface_shortwave_up,
        :SLRU  => diagn.physics.surface_longwave_up,
        :SSRD  => diagn.physics.surface_shortwave_down,
        :SLRD  => diagn.physics.surface_longwave_down,
        :OSR   => diagn.physics.outgoing_shortwave_radiation,
        :OLR   => diagn.physics.outgoing_longwave_radiation,
        :albedo=> diagn.physics.albedo
    )

    # spectral helpers (ℓ=0 → global mean; multiply by area for total)
    function calc_global_mean(field)
        a = transform(field)              # model transform -> spectral coeffs
        a00 = real(a[1])                  # index 1 == ℓ=0,m=0 (SpeedyWeather layout)
        return a00 / model.spectral_transform.norm_sphere
    end
    function calc_global_sum(field)
        mean_val = calc_global_mean(field)
        area = 4π * model.planet.radius^2
        return mean_val * area
    end

    calcfun = SumFlag ? calc_global_sum : calc_global_mean

    results = Dict{Symbol, Float64}()
    for (k, f) in fields
        try
            results[k] = Float64(calcfun(f))
        catch err
            @warn "calc_trenberth_from_diagn: could not compute $k: $err"
            results[k] = NaN
        end
    end

    # derived surface/Trenberth terms (adjust sign convention as needed)
    results[:SW_net_sfc]  = results[:SSRD] - results[:SSRU]
    results[:LW_net_sfc]  = results[:SLRD] - results[:SLRU]
    results[:surface_net] = results[:SW_net_sfc] + results[:LW_net_sfc] - results[:LHF] - results[:SHF]

    return results
end


calc_trenberth_from_diagn (generic function with 1 method)

In [33]:
# --- callback type ---
# Base.@kwdef mutable struct TrenberthCallback <: SpeedyWeather.AbstractCallback
#     timestep_counter::Int = 0
#     data::Dict{Symbol, Vector{Float64}} = Dict{Symbol, Vector{Float64}}()
#     times::Vector{Float64} = Float64[]  # elapsed seconds since simulation start
#     datetimes::Vector{DateTime} = DateTime[]  # corresponding DateTime objects
#     start_time::Float64 = 0.0  # simulation start time in seconds
#     SumFlag::Bool = false
# end

In [34]:
Base.@kwdef mutable struct TrenberthCallback <: SpeedyWeather.AbstractCallback
    timestep_counter::Int = 0
    data::Dict{Symbol, Vector{Float64}} = Dict{Symbol, Vector{Float64}}()
    times::Vector{Float64} = Float64[]          # elapsed seconds
    datetimes::Vector{DateTime} = DateTime[]    # original DateTime stamps
    start_time::Float64 = 0.0
    SumFlag::Bool = false
    var_longnames::Dict{Symbol,String} = TRENBERTH_LONGNAMES
end

TrenberthCallback

In [35]:
# Default long names for Trenberth variables
const TRENBERTH_LONGNAMES = Dict(
    :LHF => "Surface latent heat flux (W/m²)",
    :SHF => "Surface sensible heat flux (W/m²)",
    :SSRU => "Surface shortwave up (W/m²)",
    :SLRU => "Surface longwave up (W/m²)",
    :SSRD => "Surface shortwave down (W/m²)",
    :SLRD => "Surface longwave down (W/m²)",
    :OSR => "Outgoing shortwave radiation (TOA) (W/m²)",
    :OLR => "Outgoing longwave radiation (TOA) (W/m²)",
    :albedo => "Surface albedo",
    :SW_net_sfc => "Surface net shortwave (W/m²)",
    :LW_net_sfc => "Surface net longwave (W/m²)",
    :surface_net => "Surface net energy (W/m²)"
)

Dict{Symbol, String} with 12 entries:
  :surface_net => "Surface net energy (W/m²)"
  :SLRU        => "Surface longwave up (W/m²)"
  :LHF         => "Surface latent heat flux (W/m²)"
  :LW_net_sfc  => "Surface net longwave (W/m²)"
  :OLR         => "Outgoing longwave radiation (TOA) (W/m²)"
  :albedo      => "Surface albedo"
  :SSRD        => "Surface shortwave down (W/m²)"
  :SW_net_sfc  => "Surface net shortwave (W/m²)"
  :OSR         => "Outgoing shortwave radiation (TOA) (W/m²)"
  :SLRD        => "Surface longwave down (W/m²)"
  :SHF         => "Surface sensible heat flux (W/m²)"
  :SSRU        => "Surface shortwave up (W/m²)"

In [36]:
function TrenberthCallback(; vars = [:LHF,:SHF,:SSRU,:SLRU,:SSRD,:SLRD,:OSR,:OLR,:albedo,:SW_net_sfc,:LW_net_sfc,:surface_net],
                             SumFlag::Bool=false,
                             nsteps::Int=0,
                             var_longnames::Dict{Symbol,String}=TRENBERTH_LONGNAMES)
    d = Dict{Symbol, Vector{Float64}}()
    for v in vars
        d[v] = nsteps > 0 ? Vector{Float64}(undef, nsteps + 1) : Float64[]
    end
    times = nsteps > 0 ? Vector{Float64}(undef, nsteps + 1) : Float64[]
    datetimes = nsteps > 0 ? Vector{DateTime}(undef, nsteps + 1) : DateTime[]
    return TrenberthCallback(0, d, times, datetimes, 0.0, SumFlag, var_longnames)
end

TrenberthCallback

In [37]:
# Pretty-print the long names
function show_var_names(cb::TrenberthCallback)
    for (k, long) in cb.var_longnames
        println(string(k), " → ", long)
    end
    return nothing
end

# Optional: assemble a DataFrame if DataFrames.jl is installed
function to_dataframe(cb::TrenberthCallback)
    try
        @eval using DataFrames
    catch
        error("DataFrames.jl not available. Install it with `using Pkg; Pkg.add(\"DataFrames\")`")
    end
    df = DataFrame(time = cb.datetimes)
    for (k, vec) in cb.data
        colname = get(cb.var_longnames, k, string(k))  # column name uses long name if available
        # ensure column identifier is a Symbol
        df[Symbol(colname)] = vec
    end
    return df
end

to_dataframe (generic function with 1 method)

In [38]:
using Dates

# Convert various time types to Float64. Default unit = :seconds.
function time_to_float(t; unit::Symbol = :seconds)
    if t isa DateTime
        secs = Dates.datetime2unix(t)                     # seconds since Unix epoch
        return unit == :seconds ? Float64(secs) :
               unit == :days    ? Float64(secs / 86400.0) :
               error("unsupported unit: $unit")
    elseif t <: Dates.Period   # Day, Hour, Minute, etc.
        # Dates.value returns the integer magnitude in the Period's base units
        # For Day it returns number of days, for Hour number of hours, etc.
        # Convert to days or seconds depending on unit
        if unit == :days
            return float(Dates.value(t))
        elseif unit == :seconds
            # approximate: convert days/hours etc. to seconds using common ratios
            # We'll convert via Day/Hr/Minute explicitly for safety:
            if t isa Day
                return float(Dates.value(t) * 86400)
            elseif t isa Hour
                return float(Dates.value(t) * 3600)
            elseif t isa Minute
                return float(Dates.value(t) * 60)
            else
                # fallback: convert to days then seconds
                return float(Dates.value(Day(round(Int, Dates.value(t)))) * 86400)
            end
        else
            error("unsupported unit: $unit")
        end
    elseif t isa Number
        return float(t)
    else
        error("unsupported time type: $(typeof(t))")
    end
end


time_to_float (generic function with 1 method)

In [39]:
function SpeedyWeather.initialize!(cb::TrenberthCallback,
                                   progn::PrognosticVariables,
                                   diagn::DiagnosticVariables,
                                   model::AbstractModel)
    # Store the simulation start time for reference (convert DateTime to Float64 Unix timestamp)
    cb.start_time = Dates.datetime2unix(progn.clock.time)
    
    # Try to get nsteps, but if it doesn't work, just start with empty vectors
    try
        nsteps = progn.clock.nsteps
        # if our data dict vectors are empty or wrong size, (re)allocate
        for (k, v) in cb.data
            if isempty(v) || length(v) != nsteps + 1
                cb.data[k] = Vector{Float64}(undef, nsteps + 1)
            end
        end
        if isempty(cb.times) || length(cb.times) != nsteps + 1
            cb.times = Vector{Float64}(undef, nsteps + 1)
        end
        if isempty(cb.datetimes) || length(cb.datetimes) != nsteps + 1
            cb.datetimes = Vector{DateTime}(undef, nsteps + 1)
        end
    catch
        # If we can't get nsteps, just use dynamic push mode
        @info "Could not determine nsteps, using dynamic push mode"
    end

    # set counter to 1 and store initial conditions
    cb.timestep_counter = 1
    t0 = Dates.datetime2unix(progn.clock.time)  # Convert DateTime to Unix timestamp
    dt0 = progn.clock.time  # Get the original DateTime object
    # compute initial values using diagn
    res0 = calc_trenberth_from_diagn(diagn, model; SumFlag=cb.SumFlag)
    for (k, v) in res0
        if haskey(cb.data, k)
            if length(cb.data[k]) > 0
                cb.data[k][1] = v
            else
                push!(cb.data[k], v)
            end
        else
            cb.data[k] = [v]
        end
    end

    # Store time relative to simulation start (in seconds) and DateTime
    if length(cb.times) > 0
        cb.times[1] = t0 - cb.start_time
        cb.datetimes[1] = dt0
    else
        push!(cb.times, t0 - cb.start_time)
        push!(cb.datetimes, dt0)
    end
    return nothing
end

In [40]:
# --- callback! called every step (after the step completes) ---
function SpeedyWeather.callback!(cb::TrenberthCallback,
                                 progn::PrognosticVariables,
                                 diagn::DiagnosticVariables,
                                 model::AbstractModel)
    # increment step index
    cb.timestep_counter += 1
    
    # compute current diagnostics
    res = calc_trenberth_from_diagn(diagn, model; SumFlag=cb.SumFlag)
    
    # push new values to the arrays
    for (k, v) in res
        if !haskey(cb.data, k)
            # new key appeared: create vector and push
            cb.data[k] = [v]
        else
            # existing key: push to the vector
            push!(cb.data[k], v)
        end
    end
    
    # record model time relative to simulation start (in seconds) and DateTime
    # Convert DateTime to Unix timestamp, then subtract start_time to get elapsed seconds
    current_time = Dates.datetime2unix(progn.clock.time)
    push!(cb.times, current_time - cb.start_time)
    push!(cb.datetimes, progn.clock.time)  # Store the DateTime object
    return nothing
end

In [41]:
# --- finalize (optional) ---
SpeedyWeather.finalize!(cb::TrenberthCallback, args...) = nothing

adding the calbak to the model


In [42]:
# preallocated (fast) if you know nsteps:
# cb = TrenberthCallback(SumFlag=false, grow=false, nsteps=1000)

# OR dynamic push mode (flexible):
cb = TrenberthCallback(SumFlag=false,  nsteps=0)


TrenberthCallback <: AbstractCallback
├ timestep_counter::Int64 = 0
├ data::Dict{Symbol, Vector{Float64}} = Dict(:LW_net_sfc => [], :OLR => [], :SSRD => [], :SLRD => [], :SHF => [], :SSRU => [], :SLRU => [], :LHF => [], :albedo => [], :SW_net_sfc => [], :OSR => [], :surface_net => [])
├ start_time::Float64 = 0.0
├ SumFlag::Bool = false
├ var_longnames::Dict{Symbol, String} = Dict(:surface_net => "Surface net energy (W/m²)", :SLRU => "Surface longwave up (W/m²)", :LHF => "Surface latent heat flux (W/m²)", :LW_net_sfc => "Surface net longwave (W/m²)", :OLR => "Outgoing longwave radiation (TOA) (W/m²)", :albedo => "Surface albedo", :SSRD => "Surface shortwave down (W/m²)", :SW_net_sfc => "Surface net shortwave (W/m²)", :OSR => "Outgoing shortwave radiation (TOA) (W/m²)", :SLRD => "Surface longwave down (W/m²)", :SHF => "Surface sensible heat flux (W/m²)", :SSRU => "Surface shortwave up (W/m²)")
└── arrays: times, datetimes

In [43]:
# A: add into the callbacks dict directly (works if model.callbacks is a Dict-like)
add!(model.callbacks, :trenberth => cb)

In [44]:
keys(model.callbacks)              # should include :trenberth
model.callbacks[:trenberth] === cb # should be true

true

In [45]:
sim = initialize!(model)   # this will call SpeedyWeather.initialize! on cb
run!(sim, period=Day(10))  # or your usual run invocation

┌ Info: Could not determine nsteps, using dynamic push mode
└ @ Main /home/lucy_h/speedyweather_trenberth_diagram/joining/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X36sdnNjb2RlLXJlbW90ZQ==.jl:25


In [46]:
cb.data

Dict{Symbol, Vector{Float64}} with 12 entries:
  :LW_net_sfc  => [0.0, -239.742, -238.96, -238.311, -237.911, -237.739, -237.8…
  :OLR         => [0.0, 357.659, 352.521, 350.784, 346.932, 343.66, 342.421, 34…
  :SSRD        => [0.0, 341.258, 341.252, 341.259, 341.256, 341.267, 341.243, 3…
  :SLRD        => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.…
  :SHF         => [0.0, 20.9445, 25.3991, 22.7005, 19.4452, 19.1563, 15.0698, 1…
  :SSRU        => [0.0, 28.0432, 28.3989, 29.1781, 30.0129, 30.8464, 31.6498, 3…
  :SLRU        => [0.0, 239.742, 238.96, 238.311, 237.911, 237.739, 237.803, 23…
  :LHF         => [0.0, 14.0758, 14.1463, 15.1494, 14.0292, 12.547, 9.97612, 7.…
  :albedo      => [0.0, 0.115173, 0.115173, 0.115173, 0.115173, 0.115173, 0.115…
  :SW_net_sfc  => [0.0, 313.214, 312.853, 312.081, 311.243, 310.421, 309.593, 3…
  :OSR         => [0.0, 28.0432, 28.3989, 29.1781, 30.0129, 30.8464, 31.6498, 3…
  :surface_net => [0.0, 38.4521, 34.3478, 35.9207, 39.8579, 40

In [47]:
cb.times

361-element Vector{Float64}:
      0.0
   2400.0
   4800.0
   7200.0
   9600.0
  12000.0
  14400.0
  16800.0
  19200.0
  21600.0
      ⋮
 844800.0
 847200.0
 849600.0
 852000.0
 854400.0
 856800.0
 859200.0
 861600.0
 864000.0

In [48]:
cb.datetimes

361-element Vector{DateTime}:
 2000-01-01T00:00:00
 2000-01-01T00:40:00
 2000-01-01T01:20:00
 2000-01-01T02:00:00
 2000-01-01T02:40:00
 2000-01-01T03:20:00
 2000-01-01T04:00:00
 2000-01-01T04:40:00
 2000-01-01T05:20:00
 2000-01-01T06:00:00
 ⋮
 2000-01-10T18:40:00
 2000-01-10T19:20:00
 2000-01-10T20:00:00
 2000-01-10T20:40:00
 2000-01-10T21:20:00
 2000-01-10T22:00:00
 2000-01-10T22:40:00
 2000-01-10T23:20:00
 2000-01-11T00:00:00

In [49]:
cb.var_longnames

Dict{Symbol, String} with 12 entries:
  :surface_net => "Surface net energy (W/m²)"
  :SLRU        => "Surface longwave up (W/m²)"
  :LHF         => "Surface latent heat flux (W/m²)"
  :LW_net_sfc  => "Surface net longwave (W/m²)"
  :OLR         => "Outgoing longwave radiation (TOA) (W/m²)"
  :albedo      => "Surface albedo"
  :SSRD        => "Surface shortwave down (W/m²)"
  :SW_net_sfc  => "Surface net shortwave (W/m²)"
  :OSR         => "Outgoing shortwave radiation (TOA) (W/m²)"
  :SLRD        => "Surface longwave down (W/m²)"
  :SHF         => "Surface sensible heat flux (W/m²)"
  :SSRU        => "Surface shortwave up (W/m²)"

### Creating observables from callback output

The callback outputs a dictionary with the fluxes as keys. To establish an efficient pipeline with the Makie interface, the dictionary needs to be converted to observables. These are mutable containers that can be listened to and tracked. Makie interacts with these observables and reacts when the observables change. These are ideal for integrating multiple streams of information within one diagram. 

In [ ]:
using Observables, Dates

# --- assume `cb` is your filled TrenberthCallback after a run --- #

# 1) decide which variables you want in the NamedTuple and in what order
vars_order = [:LHF, :SHF, :SSRU, :SLRU, :SSRD, :SLRD, :OSR, :OLR, :albedo,
              :SW_net_sfc, :LW_net_sfc, :surface_net]
## Lists the variable in time step order. Builds consistent tuples for plotting (
# the plots always find the same fields)

# 2) compute a safe length (use minimum so all fields exist for each index)
lengths = Int[]
for k in vars_order
    if !haskey(cb.data, k)
        error("cb.data missing variable $k")
    end
    push!(lengths, length(cb.data[k]))
end
n = minimum(lengths)            # safe common length; change to maximum if you prefill missing
## cb.data[:LHF] is a vector that the callback filled at runtime. Different vectors may have different
# lengths. Taking the minimum minimises missing entries with NaN or missing. 



# 3) build a Vector of NamedTuples, one per timestep
flux_series = Vector{NamedTuple}(undef, n)
for i in 1:n
    # extract value for each variable at time i (use cb.data[k][i])
    flux_series[i] = (
        datetime = cb.datetimes[i],   # keep the DateTime for convenience
        LHF = cb.data[:LHF][i],
        SHF = cb.data[:SHF][i],
        SSRU = cb.data[:SSRU][i],
        SLRU = cb.data[:SLRU][i],
        SSRD = cb.data[:SSRD][i],
        SLRD = cb.data[:SLRD][i],
        OSR  = cb.data[:OSR][i],
        OLR  = cb.data[:OLR][i],
        albedo = cb.data[:albedo][i],
        SW_net_sfc = cb.data[:SW_net_sfc][i],
        LW_net_sfc = cb.data[:LW_net_sfc][i],
        surface_net = cb.data[:surface_net][i]
    )
end
## Builds a NamedTuple with the same field names. NamedTuple was used because it is convenient and immutables
## Datetime included so current_point also includes the timestamp


# 4) wrap in an Observable for UI binding
history_obs = Observable(flux_series)   # Observable{Vector{NamedTuple}}
## Wraps the whole vector in one observable 
## When history_obs is replaced and observables.notify! is called, derived observables update


# 5) time index observable (for slider)
time_idx = Observable(1)                # integer index, 1..n
## Small observable integer that tracks which timestep is selected


# 6) derived observable for the current timestep (
current_point = map((hist, idx) -> hist[idx], history_obs, time_idx)
# Now current_point[] is the NamedTuple at the selected index.
## Whenever history_obs[] or time_idx[] change, current_point updates automatically


# Example: pull a scalar observable for one flux (e.g., LHF) for easy plotting
current_LHF = map(p -> p.LHF, current_point)   # Observable{Float64} representing current LHF

# Example: print when slider changes
on(current_point) do p
    @info "time = $(p.datetime), LHF = $(p.LHF), SSRD = $(p.SSRD)"
end


ObserverFunction defined at /home/lucy_h/speedyweather_trenberth_diagram/joining/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X54sdnNjb2RlLXJlbW90ZQ==.jl:55 operating on Observable((datetime = DateTime("2000-01-01T00:00:00"), LHF = 0.0, SHF = 0.0, SSRU = 0.0, SLRU = 0.0, SSRD = 0.0, SLRD = 0.0, OSR = 0.0, OLR = 0.0, albedo = 0.0, SW_net_sfc = 0.0, LW_net_sfc = 0.0, surface_net = 0.0))

### Checking observables 

These observables are scalar. Each observable holds a single numerical value (global mean flux in W/m2) at the currently selected timestep. 

The following line:  map(p -> p.LHF, current point) creates a derived observables from the scalar LHF value from the current_point observable. 

Current point --> holds the flux fields. It is not scalar, but the fields are scalar. 

In [57]:
## Testing observables are working 
println(current_point[])

┌ Info: time = 2000-01-07T00:00:00, LHF = 11.351363182067871, SSRD = 341.2523193359375
└ @ Main /home/lucy_h/speedyweather_trenberth_diagram/joining/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X54sdnNjb2RlLXJlbW90ZQ==.jl:55
┌ Info: time = 2000-01-07T00:00:00, LHF = 11.351363182067871, SSRD = 341.2523193359375
└ @ Main /home/lucy_h/speedyweather_trenberth_diagram/joining/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X54sdnNjb2RlLXJlbW90ZQ==.jl:55


(datetime = DateTime("2000-01-07T00:00:00"), LHF = 11.351363182067871, SHF = 8.479742050170898, SSRU = 27.13477897644043, SLRU = 239.36683654785156, SSRD = 341.2523193359375, SLRD = 0.0, OSR = 27.13477897644043, OLR = 323.6065673828125, albedo = 0.11521648615598679, SW_net_sfc = 314.11754035949707, LW_net_sfc = -239.36683654785156, surface_net = 54.91959857940674)


### Plotting function

Sets up derived global mean flux observables. 

In [60]:
using GLMakie, Observables, Dates

# ---------------- Assume flux_series, history_obs, time_idx, current_point are already created ----------------
nsteps = length(flux_series)

# ---------------- Extract individual flux observables ----------------
LHF_obs = map(p -> p.LHF, current_point)
SHF_obs = map(p -> p.SHF, current_point)
SSRU_obs = map(p -> p.SSRU, current_point)
SLRU_obs = map(p -> p.SLRU, current_point)
SSRD_obs = map(p -> p.SSRD, current_point)
SLRD_obs = map(p -> p.SLRD, current_point)
OSR_obs  = map(p -> p.OSR,  current_point)
OLR_obs  = map(p -> p.OLR,  current_point)
SW_net_sfc_obs = map(p -> p.SW_net_sfc, current_point)



Observable(313.6764545440674)


#### Visual setup of figure

In [61]:
# ---------------- Visual Setup ----------------
fig = Figure(size = (1400, 800), backgroundcolor = :white)
ax = Axis(fig[1, 1:3], aspect = DataAspect(), backgroundcolor = (:lightblue, 0.15))
hidedecorations!(ax)
hidespines!(ax)
xlims!(ax, -3, 3)
ylims!(ax, -2, 2.5)

#### Arrow drawing function

In [ ]:
# ---------------- Arrow drawing (shaft + head) with length_scale ----------------
function draw_arrow!(ax, x, y_start, y_end, flux_obs, label, color;
                     scale=400.0,
                     labelside=:left,
                     length_scale::Float64 = 1.0,
                     width_multiplier::Float64 = 1.0,
                     label_offset::Float64 = 0.0,   # horizontal nudge (axis units)
                     label_voffset::Float64 = 0.0)  # vertical nudge (axis units)

    # width observable (controls shaft thickness)
    width_obs = map(flux_obs) do f
        # base factor (tweak 6.0 to change baseline sensitivity)
        base = abs(f) / scale * 6.0
        # apply global multiplier and clamp to pixels range
        clamp(base * width_multiplier, 1.5, 24.0)   # increased upper bound allowed
    end

    # head length in axis units (direction aware)
    head_len = 0.12 * sign(y_end - y_start)

    # y position of the arrow tip after applying length_scale
    # compute y_tip as lift so it updates reactively if length_scale changes (rare)
    y_tip_obs = lift(width_obs) do _
        y_start + (y_end - y_start) * length_scale
    end

    # shaft end should connect to head base (y_tip - head_len)
    shaft_y = lift(y_tip_obs) do ytip
        [y_start, ytip - head_len]
    end

    # draw shaft (reactive linewidth)
    lines!(ax, [x, x], shaft_y, linewidth = width_obs, color = color)

    # head width scales with shaft width (converted into axis-units; tune multiplier)
    head_w = map(width_obs) do lw
        clamp(lw * 0.012 * (1/width_multiplier), 0.05, 0.35)
    end

    # arrow head attached to y_tip (no gap)
    arrow_head = lift(head_w, y_tip_obs) do w, ytip
        [
            Point2f(x - w, ytip - head_len),
            Point2f(x + w, ytip - head_len),
            Point2f(x,       ytip)
        ]
    end

    poly!(ax, arrow_head, color = color, strokecolor = :black, strokewidth = 1.2)

    # compute label position: midpoint of shaft (following scaled tip) + nudges
    label_mid_y = lift(y_tip_obs) do ytip
        (y_start + ytip) / 2
    end

    # horizontal label placement: base offset + user-supplied nudge
    base_horiz = labelside == :left ? (x - 0.4) : (x + 0.4)
    label_x = map(label_mid_y) do _
        base_horiz + label_offset
    end

    # vertical label position includes optional vertical offset
    label_y = lift(label_mid_y) do my
        my + label_voffset
    end

    # place text (reactive)
    lift(label_x, label_y, flux_obs) do lx, ly, f
        text!(ax, lx, ly, text = "$label\n$(round(Int, abs(f)))",
              fontsize = 10, align = (:center, :center))
    end

    return nothing
end


draw_arrow! (generic function with 1 method)

#### Drawing atmosphere and surface boxes

In [63]:
# ---------------- Boxes ----------------
poly!(ax, Point2f[(-2.5,0.2),(2.5,0.2),(2.5,1.8),(-2.5,1.8)],
      color=(:skyblue,0.25), strokecolor=:steelblue, strokewidth=2)
text!(ax, 0, 1.0, text="Atmosphere", fontsize=18, align=(:center,:center))

poly!(ax, Point2f[(-2.5,-1.2),(2.5,-1.2),(2.5,0.0),(-2.5,0.0)],
      color=(:tan,0.3), strokecolor=:saddlebrown, strokewidth=2)
text!(ax, 0, -0.6, text="Surface", fontsize=18, align=(:center,:center))


Makie.Text{Tuple{Vector{Point{2, Float64}}}}

#### Drawing arrows for fluxes

In [64]:
# ---------------- Arrows ----------------
draw_arrow!(ax, -2.0,  2.2,  1.9, SSRD_obs, "SW Down", :gold)
draw_arrow!(ax, -1.3,  1.9,  2.2, SSRU_obs, "Reflected", :yellow)

# Sensible (shorter)
draw_arrow!(ax, -0.7, -0.1, 1.6, SHF_obs, "Sensible", :darkcyan;
            scale = 150.0, labelside = :right, length_scale = 0.65)

# Latent (shorter)
draw_arrow!(ax, 0.0, -0.1, 1.6, LHF_obs, "Latent", :dodgerblue;
            scale = 200.0, labelside = :right, length_scale = 0.65)

draw_arrow!(ax,  0.7,  1.6, -0.1, SLRD_obs, "LW Down",  :orangered)
draw_arrow!(ax,  1.4, -0.1,  1.6, SLRU_obs, "LW Up",    :red)
draw_arrow!(ax, -2.0,  0.1, -1.0, SW_net_sfc_obs, "SW Absorbed", :orange)
draw_arrow!(ax,  2.0,  1.9,  2.2, OLR_obs,  "OLR", :darkred)
draw_arrow!(ax, -0.5,  1.9,  2.2, OSR_obs,  "OSR", :gold)


In [ ]:
# ---------------- Slider ----------------
control_layout = GridLayout(fig[2, 1:3])
slider = Slider(control_layout[1,1], range = 1:nsteps, startvalue = 1)

on(slider.value) do v
    time_idx[] = Int(round(v))
end
on(time_idx) do i
    set_close_to!(slider, i)
end

# ---------------- Buttons (Observable labels) ----------------
play_label  = Observable("Play")
reset_label = Observable("Reset")

play_button  = Button(control_layout[2,1], label = play_label)
reset_button = Button(control_layout[2,2], label = reset_label)

# ---------------- Animation engine ----------------
is_playing = Observable(false)
anim_task = Ref{Union{Task,Nothing}}(nothing)

function start_animation!(dt=0.05)
    if anim_task[] !== nothing && !istaskdone(anim_task[])
        return
    end
    anim_task[] = @async begin
        while is_playing[]
            time_idx[] = (time_idx[] % nsteps) + 1
            sleep(dt)
            yield()
        end
    end
end

function stop_animation!()
    is_playing[] = false
    if anim_task[] !== nothing && !istaskdone(anim_task[])
        try
            @async Base.throwto(anim_task[], InterruptException())
        catch
        end
    end
end

# ---------------- Button handlers ----------------
on(play_button.clicks) do _
    is_playing[] = !is_playing[]
    if is_playing[]
        play_label[] = "Pause"
        start_animation!(0.05)
    else
        play_label[] = "Play"
        stop_animation!()
    end
end

on(reset_button.clicks) do _
    stop_animation!()
    play_label[] = "Play"
    time_idx[] = 1
end

# ---------------- Auto-play on launch ----------------
is_playing[] = true
play_label[] = "Pause"
start_animation!(0.05)

display(fig)

┌ Info: time = 2000-01-01T00:00:00, LHF = 0.0, SSRD = 0.0
└ @ Main /home/lucy_h/speedyweather_trenberth_diagram/joining/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X54sdnNjb2RlLXJlbW90ZQ==.jl:55
┌ Info: time = 2000-01-01T00:40:00, LHF = 14.075798988342285, SSRD = 341.2576599121094
└ @ Main /home/lucy_h/speedyweather_trenberth_diagram/joining/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X54sdnNjb2RlLXJlbW90ZQ==.jl:55
┌ Info: time = 2000-01-01T00:40:00, LHF = 14.075798988342285, SSRD = 341.2576599121094
└ @ Main /home/lucy_h/speedyweather_trenberth_diagram/joining/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X54sdnNjb2RlLXJlbW90ZQ==.jl:55
┌ Info: time = 2000-01-01T01:20:00, LHF = 14.146276473999023, SSRD = 341.2518310546875
└ @ Main /home/lucy_h/speedyweather_trenberth_diagram/joining/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X54sdnNjb2RlLXJlbW90ZQ==.jl:55
┌ Info: time = 2000-01-01T01:20:00, LHF = 14.146276473999023, SSRD = 341.2518310546875
└ @ Main /home/lucy_h/

GLMakie.Screen(...)

In [66]:
## Checking animation output
# 1) Check animation state
@show isdefined(Main, :anim_task) ? anim_task[] : "no anim_task var"
@show isdefined(Main, :is_playing) ? is_playing[] : "no is_playing var"

# 2) Inspect play_button shape and properties (useful if API changed)
@show typeof(play_button)
@show propertynames(play_button)

# 3) See if multiple figures are open (quick check — shows current figure)
@show fig isa Makie.Figure


if isdefined(Main, :anim_task)
    anim_task[]
else
    "no anim_task var"
end = Task (failed) @0x00007a7746635d20
if isdefined(Main, :is_playing)
    is_playing[]
else
    "no is_playing var"
end = false
typeof(play_button) = Button
propertynames(play_button) = (:parent, :layoutobservables, :blockscene, :halign, :valign, :padding, :fontsize, :label, :font, :width, :height, :tellwidth, :tellheight, :cornerradius, :cornersegments, :strokewidth, :strokecolor, :buttoncolor, :labelcolor, :labelcolor_hover, :labelcolor_active, :buttoncolor_active, :buttoncolor_hover, :clicks, :alignmode)
fig isa Makie.Figure = true


true